In [1]:
"""
=======================================================================
 NumPy Walkthrough: Analyzing the Pima Indians Diabetes Dataset
=======================================================================
Dataset : Pima Indians Diabetes Database (public, UCI ML Repository)
Source  : https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv
Rows    : 768 female patients (Pima Indian heritage, age 21+)
Columns : 8 clinical measurements + 1 outcome label (0 = non-diabetic, 1 = diabetic)

This script is written as a FOLLOW-ALONG teaching example. Every step is
numbered and explained in plain language before the code runs it, so a
pharmacy student with no prior programming background can read top to
bottom and understand not just WHAT each line does, but WHY.

Only NumPy is used for the numerical analysis (no pandas), so the focus
stays on core array operations: loading, indexing, boolean filtering,
and summary statistics.
=======================================================================
"""

import numpy as np
import urllib.request
import os

# -----------------------------------------------------------------------
# STEP 1: Download the dataset (only if we don't already have it)
# -----------------------------------------------------------------------
# The dataset is a plain CSV file with NO header row. Each line is one
# patient, with 9 comma-separated numbers.
# The below code checks if a local diabeters.csv file exists or not
# local copy available --> Then use it
# Else download it from Github & store it as local file and then use it for further analysis.
# CSV means Comma Seperated Values (each value is seperated by a Comma)
DATA_URL = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
LOCAL_FILE = "diabetes.csv"

if not os.path.exists(LOCAL_FILE):
    print("Downloading dataset...")
    urllib.request.urlretrieve(DATA_URL, LOCAL_FILE)
    print("Download complete.\n")
else:
    print(f"Using already-downloaded file: {LOCAL_FILE}\n")

# -----------------------------------------------------------------------
# STEP 2: Load the CSV into a NumPy array
# -----------------------------------------------------------------------
# np.loadtxt() reads a text file straight into a 2D NumPy array.
# delimiter="," tells it the values are comma-separated.
data = np.loadtxt(LOCAL_FILE, delimiter=",")

print("STEP 2: Data loaded into a NumPy array")
print("-" * 60)
print(f"Shape of the array (rows, columns): {data.shape}")
print("  -> 768 patients, 9 columns (8 measurements + 1 outcome)\n")

# -----------------------------------------------------------------------
# STEP 3: Give the columns meaningful names
# -----------------------------------------------------------------------
# NumPy arrays don't have column labels like a spreadsheet, so we keep
# a separate list of names in the SAME ORDER as the columns in the file.
column_names = [
    "Pregnancies",              # column 0
    "Glucose",                  # column 1  (plasma glucose concentration)
    "BloodPressure",            # column 2  (diastolic, mm Hg)
    "SkinThickness",            # column 3  (triceps skin fold, mm)
    "Insulin",                  # column 4  (2-hour serum insulin, mu U/ml)
    "BMI",                      # column 5  (body mass index)
    "DiabetesPedigreeFunction", # column 6  (genetic risk score)
    "Age",                      # column 7  (years)
    "Outcome",                  # column 8  (0 = non-diabetic, 1 = diabetic)
]

print("STEP 3: Column reference")
print("-" * 60)
for i, name in enumerate(column_names):
    print(f"  Column {i}: {name}")
print()

# -----------------------------------------------------------------------
# STEP 4: Split the array into features (X) and outcome label (y)
# -----------------------------------------------------------------------
# Slicing syntax: data[:, 0:8] means "all rows, columns 0 through 7"
# data[:, 8]      means "all rows, just column 8"
X = data[:, 0:8]   # the 8 clinical measurements
y = data[:, 8]     # the diabetes outcome (0 or 1) for each patient

print("STEP 4: Split into measurements (X) and outcome (y)")
print("-" * 60)
print(f"X shape: {X.shape}  (768 patients x 8 measurements)")
print(f"y shape: {y.shape}  (768 outcome labels)\n")

# -----------------------------------------------------------------------
# STEP 5: Basic descriptive statistics across the WHOLE dataset
# -----------------------------------------------------------------------
# axis=0 tells NumPy "calculate down each column" (i.e. per measurement),
# rather than across each row (per patient).
print("STEP 5: Descriptive statistics for each measurement")
print("-" * 60)
print(f"{'Measurement':<26}{'Mean':>10}{'Std Dev':>10}{'Min':>8}{'Max':>8}")
for i, name in enumerate(column_names[:-1]):   # skip Outcome column
    col = X[:, i]
    print(f"{name:<26}{np.mean(col):>10.2f}{np.std(col):>10.2f}"
          f"{np.min(col):>8.1f}{np.max(col):>8.1f}")
print()

# -----------------------------------------------------------------------
# STEP 6: Data quality check — this dataset uses 0 as a placeholder for
# MISSING values in columns where a true zero is medically impossible
# (nobody has a blood pressure or BMI of exactly 0).
# -----------------------------------------------------------------------
print("STEP 6: Data quality check — impossible zero values (likely missing data)")
print("-" * 60)
cols_to_check = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
for name in cols_to_check:
    idx = column_names.index(name)
    zero_count = np.sum(X[:, idx] == 0)          # np.where-style boolean count
    pct = (zero_count / X.shape[0]) * 100
    print(f"  {name:<16}: {zero_count:>4} zero entries  ({pct:5.1f}% of patients)")
print("  -> In real analysis, these would be treated as missing (NaN),")
print("     not as valid clinical readings, before running statistics.\n")

# -----------------------------------------------------------------------
# STEP 7: Compare diabetic vs non-diabetic patients using Boolean indexing
# -----------------------------------------------------------------------
# y == 1 creates a Boolean array (True/False for every patient).
# Using that Boolean array inside [ ] filters the rows that match.
diabetic = X[y == 1]        # rows where the outcome is 1 (diabetic)
non_diabetic = X[y == 0]    # rows where the outcome is 0 (non-diabetic)

print("STEP 7: Diabetic vs. Non-diabetic group comparison")
print("-" * 60)
print(f"Diabetic patients:     {diabetic.shape[0]}")
print(f"Non-diabetic patients: {non_diabetic.shape[0]}\n")

compare_cols = ["Glucose", "BMI", "Age", "Pregnancies"]
print(f"{'Measurement':<16}{'Diabetic Avg':>16}{'Non-Diabetic Avg':>20}{'Difference':>14}")
for name in compare_cols:
    idx = column_names.index(name)
    d_avg = np.mean(diabetic[:, idx])
    n_avg = np.mean(non_diabetic[:, idx])
    print(f"{name:<16}{d_avg:>16.2f}{n_avg:>20.2f}{(d_avg - n_avg):>14.2f}")
print()

# -----------------------------------------------------------------------
# STEP 8: Flag high-risk patients using np.where()
# -----------------------------------------------------------------------
# np.where(condition, value_if_true, value_if_false) works like a
# column-by-column if/else, applied to the whole array at once.
glucose = X[:, column_names.index("Glucose")]
risk_flag = np.where(glucose > 140, "HIGH", "NORMAL")   # simple clinical threshold

high_risk_count = np.sum(risk_flag == "HIGH")
print("STEP 8: Flagging high glucose readings (> 140 mg/dL)")
print("-" * 60)
print(f"Patients flagged HIGH glucose: {high_risk_count} out of {len(glucose)} "
      f"({(high_risk_count/len(glucose))*100:.1f}%)\n")

# -----------------------------------------------------------------------
# STEP 9: Correlation between Glucose and Outcome
# -----------------------------------------------------------------------
# np.corrcoef() returns a matrix; the value at [0,1] is the correlation
# between the two variables we passed in. Ranges from -1 to 1.
correlation_matrix = np.corrcoef(glucose, y)
glucose_outcome_corr = correlation_matrix[0, 1]

print("STEP 9: Correlation between Glucose level and Diabetes Outcome")
print("-" * 60)
print(f"Correlation coefficient: {glucose_outcome_corr:.3f}")
print("  -> Closer to +1 means higher glucose is strongly associated")
print("     with a positive diabetes outcome in this dataset.\n")

# -----------------------------------------------------------------------
# STEP 10: Percentiles — useful for setting clinical reference ranges
# -----------------------------------------------------------------------
bmi = X[:, column_names.index("BMI")]
p25, p50, p75, p95 = np.percentile(bmi, [25, 50, 75, 95])

print("STEP 10: BMI percentiles across all patients")
print("-" * 60)
print(f"25th percentile: {p25:.1f}")
print(f"50th percentile (median): {p50:.1f}")
print(f"75th percentile: {p75:.1f}")
print(f"95th percentile: {p95:.1f}\n")

# -----------------------------------------------------------------------
# STEP 11: Summary printout
# -----------------------------------------------------------------------
print("=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"Total patients analyzed : {data.shape[0]}")
print(f"Diabetic                : {diabetic.shape[0]} "
      f"({diabetic.shape[0]/data.shape[0]*100:.1f}%)")
print(f"Non-diabetic             : {non_diabetic.shape[0]} "
      f"({non_diabetic.shape[0]/data.shape[0]*100:.1f}%)")
print(f"Avg glucose (diabetic)   : {np.mean(diabetic[:, 1]):.1f} mg/dL")
print(f"Avg glucose (non-diabetic): {np.mean(non_diabetic[:, 1]):.1f} mg/dL")
print(f"Glucose–Outcome correlation: {glucose_outcome_corr:.3f}")
print("=" * 60)

Download complete.

STEP 2: Data loaded into a NumPy array
------------------------------------------------------------
Shape of the array (rows, columns): (768, 9)
  -> 768 patients, 9 columns (8 measurements + 1 outcome)

STEP 3: Column reference
------------------------------------------------------------
  Column 0: Pregnancies
  Column 1: Glucose
  Column 2: BloodPressure
  Column 3: SkinThickness
  Column 4: Insulin
  Column 5: BMI
  Column 6: DiabetesPedigreeFunction
  Column 7: Age
  Column 8: Outcome

STEP 4: Split into measurements (X) and outcome (y)
------------------------------------------------------------
X shape: (768, 8)  (768 patients x 8 measurements)
y shape: (768,)  (768 outcome labels)

STEP 5: Descriptive statistics for each measurement
------------------------------------------------------------
Measurement                     Mean   Std Dev     Min     Max
Pregnancies                     3.85      3.37     0.0    17.0
Glucose                       120.89     3

In [4]:
import numpy as np

# Creating arrays:
concentrations = np.array([100, 80, 64, 51, 41, 33, 26])  # mg/L over 7 hours
time_hours     = np.array([0, 1, 2, 3, 4, 5, 6])

# Array information:
print('Shape:', concentrations.shape)    # (7,) — 7 elements, 1 dimension
print('Size:', concentrations.size)      # 7 — total number of elements
print('Data type:', concentrations.dtype) # int64

# Element-wise operations — applied to EVERY element at once:
# (This would require a loop with regular Python lists)
conc_float = concentrations.astype(float)  # Convert to float
half_conc  = conc_float / 2               # Divide all by 2
log_conc   = np.log(conc_float)           # Natural log of all values

print('Half concentrations:', half_conc)
# [50.  40.  32.  25.5  20.5  16.5  13. ]

# Statistical functions:
print("############### Print the statistical Measures of the data now ####################")
print(f'Mean: {np.mean(concentrations):.1f} mg/L')
print(f'Std Dev: {np.std(concentrations):.1f} mg/L')
print(f'Max: {np.max(concentrations)} mg/L')
print(f'Min: {np.min(concentrations)} mg/L')




Shape: (7,)
Size: 7
Data type: int64
Half concentrations: [50.  40.  32.  25.5 20.5 16.5 13. ]
############### Print the statistical Measures of the data now ####################
Mean: 56.4 mg/L
Std Dev: 24.7 mg/L
Max: 100 mg/L
Min: 26 mg/L
